In [1]:
import re
import pdfplumber
import pandas as pd
import numpy as np

def extract_patent_data_from_pdf(pdf_path):
    all_data = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            # Patent No: after (10)
            patent_no_match = re.search(r'\(10\)\s*Patent No\.?:?\s*([^\n]+)', text)
            patent_no = patent_no_match.group(1).strip() if patent_no_match else np.nan

            # Title: after (54), up to next (two digits) or end of text
            title_match = re.search(r'\(54\)\s*(.+?)(?=\(\d{2}\)|$)', text, re.DOTALL)
            title = title_match.group(1).strip() if title_match else np.nan
            if isinstance(title, str):
                title = re.sub(r'\s+', ' ', title)

            # Inventors: after (75)
            inventors_match = re.search(r'\(75\)\s*Inventors?:?\s*([^\n]+)', text)
            inventors = inventors_match.group(1).strip() if inventors_match else np.nan

            # Filing Date (Date of Patent): after (45)
            date_match = re.search(r'\(45\)\s*Date of Patent:\s*([^\n]+)', text)
            filing_date = date_match.group(1).strip() if date_match else np.nan

            # Optional: Assignee, often after (73)
            assignee_match = re.search(r'\(73\)\s*Assignee:?\s*([^\n]+)', text)
            assignee = assignee_match.group(1).strip() if assignee_match else np.nan

            # Optional: Abstract after (57)
            abstract_match = re.search(r'\(57\)\s*(.+?)(?=\(\d{2}\)|$)', text, re.DOTALL)
            abstract = abstract_match.group(1).strip() if abstract_match else np.nan
            if isinstance(abstract, str):
                abstract = re.sub(r'\s+', ' ', abstract)

            all_data.append({
                "Patent No": patent_no,
                "Title": title,
                "Inventor(s)": inventors,
                "Assignee": assignee,
                "Filing Date": filing_date,
                "Abstract": abstract
            })

    df = pd.DataFrame(all_data)
    df.dropna(how='all', subset=["Patent No", "Title", "Inventor(s)", "Filing Date"], inplace=True)

    return df

pdf_path = r"C:\Users\NIRANJAN_REDDY\python\PDF\Patent.pdf"
df = extract_patent_data_from_pdf(pdf_path)
print(df)


         Patent No                          Title  \
0  US 7.878,787 B2  FORMING TOOL FOR MAKING FIBRE   

                         Inventor(s)  \
0  Björn Nilsson, Kimstad (SE); Lars   

                                            Assignee   Filing Date  \
0  PAKIT International Trading 4,272,318 A * 6/19...  Feb. 1, 2011   

                                            Abstract  
0  ABSTRACT Jan. 18, 2006 (SE) .....................  
